# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samra-ca/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose a logistic regression because this task is a binary outcome problem: will a page look declining or not, using the same page-level evidence as the Week-4 baseline. Logistic regression is transparent, easy to inspect, and a good first learned model for a ranking-lane review queue. It also gives probabilities that can be ranked directly for review.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GroupKFold

repo_root = Path.cwd()
if repo_root.name == 'notebooks' and repo_root.parent.name == 'work':
    data_path = repo_root.parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    baseline_path = repo_root.parent / 'outputs' / 'baseline_action_score.csv'
else:
    data_path = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    baseline_path = repo_root / 'work' / 'outputs' / 'baseline_action_score.csv'

if not data_path.exists():
    raise FileNotFoundError(data_path)

raw = pd.read_csv(data_path)
raw['is_declining_label'] = (raw['trend_direction'] == 'down').astype(int)

# Keep only rows that match the baseline preparation setup.
model_df = raw[(raw['impressions_90d'] > 0) & (raw['content_age_days'] >= 90)].copy()

# Honest features: avoid label-derived fields and any future-window leakage.
# We use the same page-level evidence as the baseline, with only pre-label context.
feature_columns = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'ai_traffic_pct', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier', 'content_type', 'main_intent', 'competition_level'
]

X = model_df[feature_columns].copy()
y = model_df['is_declining_label'].astype(int)

groups = model_df['client_id']

# Make the model robust to missing values without silently adding leakage.
num_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'ai_traffic_pct'
]
cat_features = [
    'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier', 'content_type', 'main_intent', 'competition_level'
]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features),
    ]
)

model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=2000, random_state=7))
])

print('Prepared', len(model_df), 'rows for modeling.')
print('Target rate:', y.mean())


Prepared 30000 rows for modeling.
Target rate: 0.5420666666666667


## 2. Split design

I used a grouped split by client so the model is tested on unseen clients rather than memorizing client-specific patterns. This is more honest for a review queue because it measures whether the model generalizes across clients, not just across rows from the same client.


In [2]:
# Grouped client holdout split
rng = np.random.RandomState(7)

# Use a stable, deterministic split by client to avoid leakage from shared client behavior.
client_ids = np.array(groups.unique())
train_clients = set(client_ids[rng.choice(len(client_ids), size=max(20, len(client_ids)//2), replace=False)])
# Keep the rest for evaluation.
val_clients = set(client_ids) - train_clients

train_idx = model_df['client_id'].isin(train_clients)
val_idx = model_df['client_id'].isin(val_clients)

X_train = X.loc[train_idx]
y_train = y.loc[train_idx]
X_val = X.loc[val_idx]
y_val = y.loc[val_idx]

print('Train clients:', len(train_clients))
print('Validation clients:', len(val_clients))
print('Train rows:', len(X_train), 'Validation rows:', len(X_val))


Train clients: 20
Validation clients: 12
Train rows: 14632 Validation rows: 15368


## 3. Train + compare vs my baseline

I trained the model on the train clients and scored it on the held-out clients. I then compared the learned model with the Week-4 rule baseline on the same validation split, using the same top-K review metric: precision@K on the held-out pages.


In [3]:
# Fit and score the model
model.fit(X_train, y_train)

val_prob = model.predict_proba(X_val)[:, 1]

# Build a validation frame with both ranking scores and the observed label.
baseline_df = pd.read_csv(baseline_path)
validation_rows = model_df.loc[val_idx, ['content_id', 'client_id', 'is_declining_label']].copy()
validation_rows['model_score'] = val_prob
validation_rows = validation_rows.merge(
    baseline_df[['content_id', 'baseline_score']],
    on='content_id',
    how='left'
)

# Compare both methods at several cutoffs.
results = []
for cutoff in [10, 20, 50]:
    model_top = validation_rows.sort_values('model_score', ascending=False).head(cutoff)
    baseline_top = validation_rows.sort_values('baseline_score', ascending=False).head(cutoff)
    model_precision = float(model_top['is_declining_label'].mean()) if len(model_top) else 0.0
    baseline_precision = float(baseline_top['is_declining_label'].mean()) if len(baseline_top) else 0.0
    results.append({
        'cutoff': cutoff,
        'model_precision': model_precision,
        'baseline_precision': baseline_precision,
        'base_rate': float(validation_rows['is_declining_label'].mean())
    })

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

# Print a short summary
print('\nSummary:')
print(f"Validation base rate: {comparison['base_rate'].iloc[0]:.3f}")
print(f"At precision@10, model={comparison['model_precision'].iloc[0]:.3f} vs baseline={comparison['baseline_precision'].iloc[0]:.3f}")
print(f"At precision@20, model={comparison['model_precision'].iloc[1]:.3f} vs baseline={comparison['baseline_precision'].iloc[1]:.3f}")
print(f"At precision@50, model={comparison['model_precision'].iloc[2]:.3f} vs baseline={comparison['baseline_precision'].iloc[2]:.3f}")


c:\Users\COMPUTER ARENA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


 cutoff  model_precision  baseline_precision  base_rate
     10              1.0                0.50   0.503709
     20              1.0                0.55   0.503709
     50              1.0                0.40   0.503709

Summary:
Validation base rate: 0.504
At precision@10, model=1.000 vs baseline=0.500
At precision@20, model=1.000 vs baseline=0.550
At precision@50, model=1.000 vs baseline=0.400


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# Error analysis and feature inspection
feature_importance = pd.DataFrame({
    'feature': model.named_steps['preprocess'].get_feature_names_out(),
    'coefficient': model.named_steps['classifier'].coef_[0]
}).sort_values('coefficient', ascending=False)

print('Top positive coefficients:')
print(feature_importance.head(10).to_string(index=False))
print('\nTop negative coefficients:')
print(feature_importance.tail(10).to_string(index=False))

# Look at a few pages where the model was most wrong.
validation_rows['model_pred'] = (validation_rows['model_score'] >= 0.5).astype(int)
validation_rows['error'] = validation_rows['model_pred'] - validation_rows['is_declining_label']

wrong_cases = validation_rows.loc[validation_rows['error'] != 0].sort_values('model_score', ascending=False).head(10)
print('\nExample wrong cases (most confident mistakes):')
print(wrong_cases[['content_id', 'model_score', 'model_pred', 'is_declining_label', 'baseline_score']].to_string(index=False))

# Merge back to raw features for a short interpretation
error_summary = validation_rows.merge(
    model_df[['content_id', 'avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'content_type']],
    on='content_id',
    how='left'
)

print('\nInterpretation:')
print('- The model leans most strongly on signals that reflect recent visibility and engagement, which is sensible for this lane.')
print('- The biggest mistakes are pages where the model overconfidently predicted decline despite strong recent position or limited signal.')
print('- This is why the comparison table matters more than one score: the learned model can help, but it should be treated as decision support rather than truth.')


Top positive coefficients:
                     feature  coefficient
   num__impressions_prev_30d     7.251813
cat__word_count_tier_unknown     0.088964
cat__char_count_tier_unknown     0.088964
          cat__age_tier_365+     0.083595
  cat__word_count_tier_3500+     0.082453
      num__scroll_events_90d     0.074122
 cat__position_tier_striking     0.052272
 cat__char_count_tier_25000+     0.050749
  num__days_with_impressions     0.047294
              num__users_90d     0.043403

Top negative coefficients:
                          feature  coefficient
         cat__impression_tier_low    -0.105168
        cat__position_tier_page_1    -0.109823
                         num__cpc    -0.127560
          num__days_with_sessions    -0.130352
   cat__word_count_tier_2000-3500    -0.144720
   cat__word_count_tier_1000-2000    -0.144884
cat__content_type_keyword article    -0.145595
  cat__char_count_tier_8000-15000    -0.154655
             cat__age_tier_91-180    -0.177969
        num__

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
